In [1]:
import pandas as pd

df = pd.read_csv("bestellungen_maerz_roh.csv")

df.head()

,bestell_id,datum,kunde_email,land,produkt,menge,preis_eur
0,2032,2026-03-20,joerg.schaefer@example.com,AT,Dockingstation,4,"129,00"
1,2021,15.03.2026,max.weiss@example.com,DE,Maus,2,19.90
2,2048,2026-03-28,timo.jaeger@example.com,DE,Laptopständer,2,24.95
3,2018,2026-03-14,lena.krueger@example.com,DE,Webcam HD,2,"49,50"
4,2011,2026-03-10,max.weiss@example.com,de,Headset,4,"59,90"


In [2]:
print("Zeilen und Spalten:", df.shape)

print("\nDatentypen:")
print(df.dtypes)

print("\nFehlende Werte:")
print(df.isnull().sum())

Zeilen und Spalten: (60, 7)

Datentypen:
bestell_id     int64
datum            str
kunde_email      str
land             str
produkt          str
menge          int64
preis_eur        str
dtype: object

Fehlende Werte:
bestell_id     0
datum          0
kunde_email    5
land           0
produkt        0
menge          0
preis_eur      0
dtype: int64


In [3]:
df["preis_eur"] = pd.to_numeric(
    df["preis_eur"].str.replace(",", ".", regex=False),
    errors="coerce"
)

print("Datentyp:", df["preis_eur"].dtype)
print("Nicht umwandelbare Preise:", df["preis_eur"].isna().sum())

df[["bestell_id", "preis_eur"]].head(10)

Datentyp: float64
Nicht umwandelbare Preise: 0


,bestell_id,preis_eur
0,2032,129.00
1,2021,19.90
2,2048,24.95
3,2018,49.50
4,2011,59.90
5,2045,19.90
6,2035,24.95
7,2016,49.50
8,2022,19.90
9,2031,24.95


In [4]:
df["datum_original"] = df["datum"]

df["datum"] = pd.to_datetime(
    df["datum"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

print("Datentyp:", df["datum"].dtype)
print("Ungültige Datumswerte:", df["datum"].isna().sum())

df[["bestell_id", "datum_original", "datum"]].head(10)

Datentyp: datetime64[us]
Ungültige Datumswerte: 0


,bestell_id,datum_original,datum
0,2032,2026-03-20,2026-03-20
1,2021,15.03.2026,2026-03-15
2,2048,2026-03-28,2026-03-28
3,2018,2026-03-14,2026-03-14
4,2011,2026-03-10,2026-10-03
5,2045,27.03.2026,2026-03-27
6,2035,2026-03-22,2026-03-22
7,2016,2026-03-12,2026-12-03
8,2022,2026-03-15,2026-03-15
9,2031,2026-03-20,2026-03-20


In [5]:
def datum_korrigieren(wert):
    if "." in wert:
        return pd.to_datetime(wert, format="%d.%m.%Y")
    else:
        return pd.to_datetime(wert, format="%Y-%m-%d")

df["datum"] = df["datum_original"].apply(datum_korrigieren)

print("Ungültige Datumswerte:", df["datum"].isna().sum())

df[["bestell_id", "datum_original", "datum"]].head(10)

Ungültige Datumswerte: 0


,bestell_id,datum_original,datum
0,2032,2026-03-20,2026-03-20
1,2021,15.03.2026,2026-03-15
2,2048,2026-03-28,2026-03-28
3,2018,2026-03-14,2026-03-14
4,2011,2026-03-10,2026-03-10
5,2045,27.03.2026,2026-03-27
6,2035,2026-03-22,2026-03-22
7,2016,2026-03-12,2026-03-12
8,2022,2026-03-15,2026-03-15
9,2031,2026-03-20,2026-03-20


In [6]:
print("Länder vor der Bereinigung:")
print(df["land"].value_counts())

df["land"] = df["land"].replace({
    "Deutschland": "DE",
    "de": "DE",
    "Österreich": "AT",
    "at": "AT"
})

print("\nLänder nach der Bereinigung:")
print(df["land"].value_counts())

Länder vor der Bereinigung:
land
DE             33
AT             12
Deutschland     6
de              5
Österreich      3
at              1
Name: count, dtype: int64

Länder nach der Bereinigung:
land
DE    44
AT    16
Name: count, dtype: int64


In [7]:
print("Verteilung der Bestellmengen:")
print(df["menge"].value_counts().sort_index())

print("\nBestellungen mit Menge 0 oder negativ:")
print(df[df["menge"] <= 0])

print("\nDoppelte Bestell-IDs:")
print(df[df["bestell_id"].duplicated(keep=False)])

Verteilung der Bestellmengen:
menge
1    17
2    21
3     7
4    15
Name: count, dtype: int64

Bestellungen mit Menge 0 oder negativ:
Empty DataFrame
Columns: [bestell_id, datum, kunde_email, land, produkt, menge, preis_eur, datum_original]
Index: []

Doppelte Bestell-IDs:
    bestell_id      datum                 kunde_email land          produkt  \
2         2048 2026-03-28     timo.jaeger@example.com   DE    Laptopständer   
9         2031 2026-03-20     timo.jaeger@example.com   AT    Laptopständer   
16        2033 2026-03-20      eva.brandt@example.com   AT   Dockingstation   
22        2029 2026-03-20  joerg.schaefer@example.com   DE        Webcam HD   
24        2030 2026-03-20  joerg.schaefer@example.com   DE             Maus   
35        2030 2026-03-20  joerg.schaefer@example.com   DE             Maus   
37        2012 2026-03-11    lena.krueger@example.com   DE  Monitor 24 Zoll   
38        2001 2026-03-02    sophie.gross@example.com   DE   USB-C-Kabel 2m   
40        2033 

In [8]:
print("Doppelte Bestell-IDs:")
print(df["bestell_id"].duplicated().sum())

print("\nVollständig doppelte Datensätze:")
print(df.duplicated().sum())

print("\nIdentische Datensätze:")
display(df[df.duplicated(keep=False)])

Doppelte Bestell-IDs:
7

Vollständig doppelte Datensätze:
7

Identische Datensätze:


,bestell_id,datum,kunde_email,land,produkt,menge,preis_eur,datum_original
2,2048,2026-03-28,timo.jaeger@example.com,DE,Laptopständer,2,24.95,2026-03-28
9,2031,2026-03-20,timo.jaeger@example.com,AT,Laptopständer,1,24.95,2026-03-20
16,2033,2026-03-20,eva.brandt@example.com,AT,Dockingstation,2,129.00,2026-03-20
22,2029,2026-03-20,joerg.schaefer@example.com,DE,Webcam HD,1,49.50,2026-03-20
24,2030,2026-03-20,joerg.schaefer@example.com,DE,Maus,2,19.90,2026-03-20
35,2030,2026-03-20,joerg.schaefer@example.com,DE,Maus,2,19.90,2026-03-20
37,2012,2026-03-11,lena.krueger@example.com,DE,Monitor 24 Zoll,4,149.00,2026-03-11
38,2001,2026-03-02,sophie.gross@example.com,DE,USB-C-Kabel 2m,1,9.95,2026-03-02
40,2033,2026-03-20,eva.brandt@example.com,AT,Dockingstation,2,129.00,2026-03-20
47,2031,2026-03-20,timo.jaeger@example.com,AT,Laptopständer,1,24.95,2026-03-20


     ## Woche 1 – Tag 2: CSV-Datenprofil

### Untersuchte Datei
bestellungen_maerz_roh.csv

Der Datensatz enthält 60 Zeilen und 7 ursprüngliche Spalten.

### Erkannte Datenqualitätsprobleme

1. **Preisformate:** Die Preise enthielten unterschiedliche Dezimaltrennzeichen. Durch Vereinheitlichung konnten alle 60 Preise erfolgreich in numerische Werte umgewandelt werden.

2. **Datumsformate:** Die Datumsangaben lagen in unterschiedlichen Formaten vor. Bei einer ersten automatischen Umwandlung wurden einige Datumswerte falsch interpretiert. Durch eine formatspezifische Umwandlung konnte dieser Fehler korrigiert werden.

3. **Länderbezeichnungen:** Deutschland und Österreich wurden unterschiedlich bezeichnet. Nach der Bereinigung ergeben sich 44 Bestellungen aus Deutschland und 16 aus Österreich.

4. **Duplikate:** Es wurden 7 vollständig doppelte Datensätze gefunden. Diese sollten vor einer endgültigen Auswertung überprüft und gegebenenfalls entfernt werden.

### Weitere Auffälligkeit
Bei 5 Bestellungen fehlen Kunden-E-Mail-Adressen.

### Zwischenfazit
Die Daten enthalten mehrere Qualitätsprobleme, die bei einer ungeprüften Verarbeitung zu falschen Ergebnissen führen könnten.

In [9]:
import json

with open("bestellungen_api.json", "r", encoding="utf-8") as datei:
    daten = json.load(datei)

print("Datentyp:", type(daten))
print("Inhalt:")
print(json.dumps(daten, indent=2, ensure_ascii=False)[:2000])

Datentyp: <class 'dict'>
Inhalt:
{
  "status": "ok",
  "anzahl_bestellungen": 50,
  "bestellungen": [
    {
      "bestell_id": 1001,
      "datum": "2026-03-02",
      "kunde": {
        "name": "Anna Bauer",
        "firma": "Datentechnik Nord",
        "ort": "München"
      },
      "positionen": [
        {
          "produkt": "Tastatur",
          "menge": 1,
          "preis_eur": 39.9
        }
      ]
    },
    {
      "bestell_id": 1002,
      "datum": "2026-03-03",
      "kunde": {
        "name": "Jörg Schäfer",
        "firma": "Kontor 24",
        "ort": "Münster"
      },
      "positionen": [
        {
          "produkt": "Headset",
          "menge": 2,
          "preis_eur": 59.9
        },
        {
          "produkt": "Tastatur",
          "menge": 2,
          "preis_eur": 39.9
        }
      ]
    },
    {
      "bestell_id": 1003,
      "datum": "2026-03-04",
      "kunde": {
        "name": "Lena Krüger",
        "firma": "MediaPoint",
        "ort": "Würzb

In [11]:
df_json = pd.json_normalize(
    daten["bestellungen"],
    record_path="positionen",
    meta=[
        "bestell_id",
        "datum",
        ["kunde", "name"],
        ["kunde", "firma"],
        ["kunde", "ort"]
    ],
    sep="_"
)

print("Zeilen und Spalten:", df_json.shape)

df_json.head(10)

Zeilen und Spalten: (72, 8)


,produkt,menge,preis_eur,bestell_id,datum,kunde_name,kunde_firma,kunde_ort
0,Tastatur,1,39.90,1001,2026-03-02,Anna Bauer,Datentechnik Nord,München
1,Headset,2,59.90,1002,2026-03-03,Jörg Schäfer,Kontor 24,Münster
2,Tastatur,2,39.90,1002,2026-03-03,Jörg Schäfer,Kontor 24,Münster
3,Monitor 24 Zoll,3,149.00,1003,2026-03-04,Lena Krüger,MediaPoint,Würzburg
4,Laptopständer,4,24.95,1004,2026-03-05,Ömer Yilmaz,Kabelwerk,Köln
5,Webcam HD,1,49.50,1005,2026-03-06,Sophie Groß,TechHaus,Fürth
6,Laptopständer,1,24.95,1005,2026-03-06,Sophie Groß,TechHaus,Fürth
7,Maus,2,19.90,1006,2026-03-09,Max Weiß,Logistik Plus,Hamburg
8,Dockingstation,3,129.00,1007,2026-03-10,Clara Böhm,Printagentur,Düsseldorf
9,USB-C-Kabel 2m,4,9.95,1008,2026-03-11,Timo Jäger,Bürobedarf Süd,Lübeck


In [12]:
 # Anzahl der Positionen in der ursprünglichen JSON-Datei
anzahl_positionen = sum(
    len(bestellung["positionen"])
    for bestellung in daten["bestellungen"]
)

print("Positionen in JSON:", anzahl_positionen)
print("Zeilen in Tabelle:", len(df_json))

# Prüfen, ob beide Zahlen übereinstimmen
assert anzahl_positionen == len(df_json)

print("Prüfung erfolgreich: Alle Positionen übernommen!")

Positionen in JSON: 72
Zeilen in Tabelle: 72
Prüfung erfolgreich: Alle Positionen übernommen!


In [14]:
import importlib.util

print("PyArrow installiert:", importlib.util.find_spec("pyarrow") is not None)

PyArrow installiert: True


In [15]:
df.to_parquet("bestellungen_maerz_bereinigt.parquet", index=False)

df_json.to_parquet("bestellungen_api.parquet", index=False)

print("Beide Parquet-Dateien erfolgreich gespeichert!")

Beide Parquet-Dateien erfolgreich gespeichert!


In [16]:
df_test = pd.read_parquet("bestellungen_maerz_bereinigt.parquet")

print("Zeilen und Spalten:", df_test.shape)
df_test.head()

Zeilen und Spalten: (60, 8)


,bestell_id,datum,kunde_email,land,produkt,menge,preis_eur,datum_original
0,2032,2026-03-20,joerg.schaefer@example.com,AT,Dockingstation,4,129.00,2026-03-20
1,2021,2026-03-15,max.weiss@example.com,DE,Maus,2,19.90,15.03.2026
2,2048,2026-03-28,timo.jaeger@example.com,DE,Laptopständer,2,24.95,2026-03-28
3,2018,2026-03-14,lena.krueger@example.com,DE,Webcam HD,2,49.50,2026-03-14
4,2011,2026-03-10,max.weiss@example.com,DE,Headset,4,59.90,2026-03-10


In [18]:
import os

csv_groesse = os.path.getsize("bestellungen_maerz_roh.csv")
parquet_groesse = os.path.getsize("bestellungen_maerz_bereinigt.parquet")

print("CSV-Datei:", csv_groesse, "Bytes")
print("Parquet-Datei:", parquet_groesse, "Bytes")

unterschied = (1 - parquet_groesse / csv_groesse) * 100

print(f"Speicherplatzersparnis: {unterschied:.2f} %")

CSV-Datei: 3852 Bytes
Parquet-Datei: 6300 Bytes
Speicherplatzersparnis: -63.55 %


### Aufgabe 4 – Größenvergleich CSV und Parquet

Die ursprüngliche CSV-Datei ist 3.852 Bytes groß.
Die bereinigte Parquet-Datei benötigt 6.300 Bytes.

Damit ist die Parquet-Datei etwa 63,55 % größer.

Das liegt vermutlich daran, dass Parquet zusätzliche Metadaten
speichert, die bei kleinen Datensätzen stärker ins Gewicht fallen.

Bei größeren Datenmengen kann Parquet durch Komprimierung und
spaltenorientierte Speicherung effizienter sein.

Hinweis: Verglichen wurden die Rohdaten als CSV mit den
bereinigten Daten als Parquet. Die Inhalte sind daher nicht
vollständig identisch.
### Aufgabe 5 – Reflexion zum KI-gestützten Arbeiten

Die KI-Unterstützung hat mir geholfen, Python-Code zur Datenbereinigung und zur Umwandlung von JSON in Parquet zu erstellen. Besonders bei der Datumsumwandlung wurde deutlich, dass vorgeschlagener Code trotz fehlerfreier Ausführung falsche Ergebnisse liefern kann. Deshalb habe ich die Ergebnisse anhand der Originaldaten überprüft und den Code entsprechend korrigiert.